# Unified label table for `data_to_merge/`

Builds **`merged-labels.csv`**: one tidy table over every dataset folder in this directory (one row = one image, one addressed video frame, or one segmentation mask).
All logic lives in the `unify/` package - **one adapter per dataset** (`unify/adapters/`) and **one class-vocabulary CSV per dataset** (`unify/vocab/`).
This notebook only runs it, validates the result, writes the CSV and prints overview tables.

* process documentation: `MERGE_PROCESS.md` &nbsp;·&nbsp; plugging in a new dataset: `docs/ADD_A_DATASET.md` &nbsp;·&nbsp; column dictionary: `docs/SCHEMA.md` &nbsp;·&nbsp; per-dataset facts: `docs/datasets/`, `docs/reports/`
* re-run: `cd data_to_merge && ../.venv/bin/python -m unify` (same result as this notebook), or `jupyter nbconvert --to notebook --execute --inplace merge_datasets.ipynb`

## 1. Setup and parameters

In [2]:
import sys, time
from pathlib import Path

# data_to_merge/ = the folder that contains unify/ (works from data_to_merge/, from the project root, or anywhere below them)
here = Path.cwd().resolve()
DATA_ROOT = next(p for p in [here, here / "data", *here.parents] if (p / "unify" / "schema.py").exists())
sys.path.insert(0, str(DATA_ROOT))

import pandas as pd
pd.set_option("display.width", 250, "display.max_columns", 60, "display.max_rows", 400, "display.max_colwidth", 80)

from data.unify import schema, registry
from data.unify.build import build_all, write_csv, read_csv
from data.unify.validate import validate, validate_roundtrip

# ---- parameters ------------------------------------------------------------------------------------------------------
OUTPUT_CSV = DATA_ROOT / "merged-labels.csv"
PARAMS = {
    "FRAMES_PER_VIDEO": 40,   # Hyper-Kvasir clips: evenly spread frames per video (same default as merge/merge_labels.ipynb, 374 clips -> 7,480 rows)
    "FRAME_STRIDE": None,     # or an int: every n-th frame instead (stride 1 = all 875,940 frames)
}
COMPUTE_MD5 = True            # md5 of every image (cached in unify/cache/): exact-duplicate detection across the whole table
print("DATA_ROOT:", DATA_ROOT)

DATA_ROOT: /Users/henrykwolek/Studia/DatasetDestroyer/data


## 2. Which datasets will be merged?

Every `unify/adapters/<name>.py` registers one dataset. A top-level folder **without** an adapter is listed below as *not merged* - that is the signal that a new dataset is waiting to be plugged in (`docs/ADD_A_DATASET.md`).

In [3]:
adapters = registry.discover()
def expected_rows(meta):                      # META["expected"] is a dict, or a function of the build parameters
    e = meta["expected"]
    return e(PARAMS) if callable(e) else e
reg = pd.DataFrame([{
    "dataset": n, "folder": m.META["folder"], "license": m.META["license"], "procedure": m.META["procedure"], "imaging": m.META["imaging"],
    "region (default)": m.META["region"], "expected rows": ", ".join(f"{k}={v:,}" for k, v in expected_rows(m.META).items()),
} for n, m in adapters.items()])
display(reg)
print("folders in data_to_merge/ WITHOUT an adapter (not merged):", registry.unregistered_folders(DATA_ROOT, adapters) or "none")

,dataset,folder,license,procedure,imaging,region (default),expected rows
0,cvc-clinicdb,CVC-ClinicDB,research and educational use only (Hospital Clinic Barcelona / CVC),colonoscopy,white-light,lower-gi-tract,"image=612, segmentation_mask=612"
1,datacrohnipi,DataCrohnIPI,CC BY-NC-ND 2.0 (research use only),capsule-small-bowel,white-light,small-bowel,"image=3,484"
2,ead2020-train,EAD2020_train,CC BY-NC-SA 4.0,unknown,unknown,unknown,"image=2,531, segmentation_mask=643"
3,etis-laribpolypdb,ETIS-LaribPolypDB,unknown,colonoscopy,white-light,lower-gi-tract,"image=196, segmentation_mask=196"
4,hyper-kvasir,hyper-kvasir,CC BY 4.0,mixed,white-light,unknown,"image=10,662, segmentation_mask=1,000, video_frame=14,951"
5,kvasir-capsule,kvasir-capsule,unknown,capsule-whole-gi,white-light,unknown,"image=47,229"
6,polypgen-ninja,polypgen-ninja,unknown,colonoscopy,unknown,lower-gi-tract,"image=8,037, segmentation_mask=3,121"
7,polypsset,PolypsSet,unknown,colonoscopy,unknown,lower-gi-tract,"image=37,899"
8,see-ai,see-ai,unknown,capsule-small-bowel,white-light,small-bowel,"image=18,481"


folders in data_to_merge/ WITHOUT an adapter (not merged): none


## 3. Build

Per dataset the adapter reads the labels and files and returns only what the dataset knows; `unify/build.py` then fills everything generic
(file size / existence, image size, md5, `dup_group_id`, `findings_all`, `is_negative`, `finding_in_image_vocab`, `sample_id`, column order and dtypes).

In [4]:
t0 = time.time()
df, report = build_all(DATA_ROOT, params=PARAMS, compute_md5=COMPUTE_MD5)
print(f"\n{len(df):,} rows x {len(df.columns)} columns in {time.time() - t0:.0f} s")

== cvc-clinicdb
   1,224 rows {'image': 612, 'segmentation_mask': 612} in 2.1 s
== datacrohnipi
  WARNING 14 images in imgs/ have no row in CrohnIPI_description.csv and are NOT ingested: 00042.jpg, 00093.jpg, 00190.jpg, 00191.jpg, 00192.jpg, 00201.jpg, 00222.jpg, 00313.jpg, 00620.jpg, 00707.jpg, 00708.jpg, 00709.jpg, 00710.jpg, 01022.jpg
   47 near-duplicate pairs (cos >= 0.985), 67 frames with a partner in another fold
   3,484 rows {'image': 3484} in 2.4 s
== ead2020-train
  WARNING mask EAD2020_semantic_00220 has no foreground in any page (labels inherited from the image boxes)
   3,174 rows {'image': 2531, 'segmentation_mask': 643} in 9.4 s
== etis-laribpolypdb
   392 rows {'image': 196, 'segmentation_mask': 196} in 2.1 s
== hyper-kvasir
   linked 1000 segmentation images 1:1 to labelled images (1000 size candidates hashed)
   116 box edge(s) equal to the image width / height were clipped to the last pixel (inclusive convention)
   10,662 images (1000 with mask), 14,951 frames from

In [5]:
rep = pd.DataFrame({n: {**{k: v for k, v in r["by_source_type"].items()}, "seconds": r["seconds"], "warnings": len(r["warnings"]), "unused vocab rows": len(r["vocab_unused"])}
                    for n, r in report["datasets"].items()}).T.fillna(0)
display(rep)
print("\n".join(report["warnings"]) or "no warnings")
for n, r in report["datasets"].items():
    if r["vocab_unused"]:
        print(f"vocab rows of {n} that no data row used (typo or vanished class?): {r['vocab_unused']}")

,image,segmentation_mask,seconds,warnings,unused vocab rows,video_frame
cvc-clinicdb,612.0,612.0,2.1,0.0,0.0,0.0
datacrohnipi,3484.0,0.0,2.4,1.0,0.0,0.0
ead2020-train,2531.0,643.0,9.4,1.0,0.0,0.0
etis-laribpolypdb,196.0,196.0,2.1,0.0,0.0,0.0
hyper-kvasir,10662.0,1000.0,3.0,0.0,0.0,14951.0
kvasir-capsule,47229.0,0.0,3.8,0.0,0.0,0.0
polypgen-ninja,8037.0,3121.0,12.0,0.0,0.0,0.0
polypsset,37899.0,0.0,11.6,1.0,0.0,0.0
see-ai,18481.0,0.0,7.0,2.0,0.0,0.0


[datacrohnipi] 14 images in imgs/ have no row in CrohnIPI_description.csv and are NOT ingested: 00042.jpg, 00093.jpg, 00190.jpg, 00191.jpg, 00192.jpg, 00201.jpg, 00222.jpg, 00313.jpg, 00620.jpg, 00707.jpg, 00708.jpg, 00709.jpg, 00710.jpg, 01022.jpg
[ead2020-train] mask EAD2020_semantic_00220 has no foreground in any page (labels inherited from the image boxes)
[polypsset] 798 annotation XML(s) have no JPEG - no rows, their boxes are lost (val2019: 376, test2019: 422; e.g. ['val2019/Annotation/1/6.xml', 'val2019/Annotation/1/72.xml', 'val2019/Annotation/1/73.xml'])
[see-ai] all_annotation.csv 'annotation_sum' row is wrong by the class id (e.g. [931, 5988, 447, 612] vs real [931, 5987, 445, 609]): it summed the class-number row too; not used
[see-ai] zero-area YOLO box kept as the 1-pixel box yolo_to_box returns: ['image07741']


## 4. Validation

Generic checks (schema, keys, controlled vocabularies, paths exist, boxes inside the image, mask <-> image links, duplicates, CSV hygiene) plus each adapter's declared row counts (`META["expected"]`). The cell raises if any check fails.

In [6]:
validate(df, DATA_ROOT, adapters, md5_checked=COMPUTE_MD5, params=report["params"]);

PASS  columns equal SCHEMA_SPEC (names and order)
PASS  dtypes equal SCHEMA_SPEC
PASS  sample_id never null
PASS  sample_id unique
PASS  source_type in allowed set
PASS  dataset is a registered adapter
PASS  label_level matches source_type
PASS  sample_id pattern for image
PASS  sample_id pattern for segmentation_mask
PASS  sample_id pattern for video_frame
PASS  (dataset, video_id, frame_index) unique for video frames
PASS  frame_index within [0, n_frames_total-1]
PASS  sample_id of frames = frame:<dataset>:<video_id>:<frame_index:06d>
PASS  timestamp_s empty outside video_frame
PASS  n_frames_total empty outside video_frame
PASS  fps empty outside video_frame
PASS  fps_effective empty outside video_frame
PASS  duration_s empty outside video_frame
PASS  codec empty outside video_frame
PASS  frames_suspect empty outside video_frame
PASS  timestamp_note empty outside video_frame
PASS  frame_rank empty outside video_frame
PASS  frames_sampled_in_video empty outside video_frame
PASS  fram

## 5. Write the CSV and verify the round trip

In [7]:
write_csv(df, OUTPUT_CSV)
print(f"{OUTPUT_CSV.relative_to(DATA_ROOT)}: {OUTPUT_CSV.stat().st_size / 1e6:.1f} MB")
validate_roundtrip(df, OUTPUT_CSV)

merged-labels.csv: 155.3 MB
round trip OK: 149,654 rows x 104 columns, LF only, no BOM


## 6. Overview

In [8]:
print("rows by dataset x source_type")
display(pd.crosstab(df["dataset"], df["source_type"], margins=True))
print("\nimages by dataset x region")
display(pd.crosstab(df.loc[df.source_type == "image", "dataset"], df.loc[df.source_type == "image", "region"], margins=True))
print("\nimages by dataset x category")
display(pd.crosstab(df.loc[df.source_type == "image", "dataset"], df.loc[df.source_type == "image", "category"], margins=True))
print("\nprocedure x imaging (image rows)")
display(pd.crosstab(df.loc[df.source_type == "image", "procedure"], df.loc[df.source_type == "image", "imaging"], margins=True))

rows by dataset x source_type


source_type,image,segmentation_mask,video_frame,All
dataset,,,,
cvc-clinicdb,612,612,0,1224
datacrohnipi,3484,0,0,3484
ead2020-train,2531,643,0,3174
etis-laribpolypdb,196,196,0,392
hyper-kvasir,10662,1000,14951,26613
kvasir-capsule,47229,0,0,47229
polypgen-ninja,8037,3121,0,11158
polypsset,37899,0,0,37899
see-ai,18481,0,0,18481



images by dataset x region


region,lower-gi-tract,small-bowel,unknown,upper-gi-tract,All
dataset,,,,,
cvc-clinicdb,612,0,0,0,612
datacrohnipi,0,3484,0,0,3484
ead2020-train,0,0,2531,0,2531
etis-laribpolypdb,196,0,0,0,196
hyper-kvasir,7210,0,0,3452,10662
kvasir-capsule,4189,0,41502,1538,47229
polypgen-ninja,8037,0,0,0,8037
polypsset,37899,0,0,0,37899
see-ai,0,18481,0,0,18481



images by dataset x category


category,anatomical-landmarks,artefact,background,normal,pathological-findings,quality-of-mucosal-views,therapeutic-interventions,All
dataset,,,,,,,,
cvc-clinicdb,0,0,0,0,612,0,0,612
datacrohnipi,0,0,0,2124,1360,0,0,3484
ead2020-train,0,2531,0,0,0,0,0,2531
etis-laribpolypdb,0,0,0,0,196,0,0,196
hyper-kvasir,4104,0,0,0,2642,1925,1991,10662
kvasir-capsule,5719,0,0,34338,4266,2906,0,47229
polypgen-ninja,0,0,4275,0,3762,0,0,8037
polypsset,0,0,1918,0,35981,0,0,37899
see-ai,0,0,6161,0,12320,0,0,18481



procedure x imaging (image rows)


imaging,chromoscopy,nbi,unknown,white-light,All
procedure,,,,,
capsule-small-bowel,0,0,0,21965,21965
capsule-whole-gi,0,0,0,47229,47229
colonoscopy,1991,5917,40019,6027,53954
gastroscopy,0,0,0,3452,3452
unknown,0,0,2531,0,2531
All,1991,5917,42550,78673,129131


In [9]:
# WHAT is annotated: which annotation kinds each dataset provides (image rows)
img = df[df.source_type == "image"]
kinds = img.groupby("dataset")["annotation_types"].agg(lambda s: ", ".join(f"{k} ({v:,})" for k, v in s.value_counts().items()))
display(kinds.to_frame("annotation_types (image rows)"))
display(img.groupby("dataset")[["has_bbox", "has_mask", "is_negative"]].sum().astype(int).rename(columns={"is_negative": "negatives"}))

,annotation_types (image rows)
dataset,
cvc-clinicdb,mask (612)
datacrohnipi,"class (3,484)"
ead2020-train,"bbox (1,888), bbox|mask (643)"
etis-laribpolypdb,mask (196)
hyper-kvasir,"class (9,662), class|bbox|mask (1,000)"
kvasir-capsule,"class (47,229)"
polypgen-ninja,"class|mask (7,396), class (641)"
polypsset,"class|bbox (35,981), bbox (1,918)"
see-ai,"bbox (18,481)"


,has_bbox,has_mask,negatives
dataset,,,
cvc-clinicdb,612,612,0
datacrohnipi,0,0,2124
ead2020-train,2531,643,0
etis-laribpolypdb,196,196,0
hyper-kvasir,1000,1000,0
kvasir-capsule,0,0,34338
polypgen-ninja,3121,3121,4275
polypsset,35981,0,1918
see-ai,12320,0,6161


In [10]:
# WHERE and WHAT finding: every class of every dataset (images + masks counted once: image rows; video clips: frame rows)
cls = (df[df.source_type != "segmentation_mask"]
       .groupby(["dataset", "region", "site", "category", "finding_harmonized", "source_type"], dropna=False).size()
       .unstack("source_type", fill_value=0).reset_index())
display(cls)

source_type,dataset,region,site,category,finding_harmonized,image,video_frame
0,cvc-clinicdb,lower-gi-tract,colon,pathological-findings,polyps,612,0
1,datacrohnipi,small-bowel,small-bowel,normal,normal-mucosa,2124,0
2,datacrohnipi,small-bowel,small-bowel,pathological-findings,aphthoid-ulceration,251,0
3,datacrohnipi,small-bowel,small-bowel,pathological-findings,edema,149,0
4,datacrohnipi,small-bowel,small-bowel,pathological-findings,erythema,125,0
5,datacrohnipi,small-bowel,small-bowel,pathological-findings,stenosis,130,0
6,datacrohnipi,small-bowel,small-bowel,pathological-findings,ulcer,705,0
7,ead2020-train,unknown,unknown,artefact,blood-in-view,34,0
8,ead2020-train,unknown,unknown,artefact,blur,148,0
9,ead2020-train,unknown,unknown,artefact,bubbles,194,0


In [11]:
# leakage-relevant: how safely can each dataset be split?
g = df[df.source_type != "segmentation_mask"].groupby("dataset").agg(
    rows=("sample_id", "size"), groups=("group_id", "nunique"), group_kind=("group_kind", lambda s: "/".join(sorted(s.unique()))),
    group_quality=("group_quality", lambda s: "/".join(sorted(s.unique()))), official_split=("split_official", lambda s: "/".join(sorted(s.dropna().unique())) or "-"),
    dup_rows=("dup_group_id", lambda s: int(s.notna().sum())), license=("license", "first"))
display(g)
print("exact-duplicate image groups across the whole table:", df["dup_group_id"].nunique(), "| groups spanning more than one dataset:",
      int((df.dropna(subset=["dup_group_id"]).groupby("dup_group_id")["dataset"].nunique() > 1).sum()))
print("rows with qa_flags:", int(df["qa_flags"].notna().sum()))
display(df["qa_flags"].dropna().str.split("|").explode().value_counts().to_frame("rows"))

,rows,groups,group_kind,group_quality,official_split,dup_rows,license
dataset,,,,,,,
cvc-clinicdb,612,29,sequence,exact,-,0,research and educational use only (Hospital Clinic Barcelona / CVC)
datacrohnipi,3484,14,block-proxy,proxy,fold-1/fold-2/fold-3/fold-4/fold-5,0,CC BY-NC-ND 2.0 (research use only)
ead2020-train,2531,28,block-proxy/sequence,exact/proxy,-,0,CC BY-NC-SA 4.0
etis-laribpolypdb,196,1,dataset,coarse,-,6,unknown
hyper-kvasir,25613,374,none/video,exact/none,-,0,CC BY 4.0
kvasir-capsule,47229,43,video,exact,-,0,unknown
polypgen-ninja,8037,217,case/centre/sequence,coarse/exact/proxy,-,8,unknown
polypsset,37899,151,video,exact,test/train/val,923,unknown
see-ai,18481,31,block-proxy,proxy,-,242,unknown


exact-duplicate image groups across the whole table: 589 | groups spanning more than one dataset: 0
rows with qa_flags: 2549


,rows
qa_flags,
duplicate-video,926
annotation-missing,641
label-folder-mismatch,480
frames-suspect,360
split-leak-known,81
dup-box-conflict,42
dup-label-conflict,36
multi-label-duplicate-file,9


## 7. Handy slices

```python
df = read_csv(OUTPUT_CSV)                                    # schema dtypes, NA-safe (or pd.read_csv(..., keep_default_na=False, na_values=[""]))
images  = df[df.source_type == "image"]                     # one row per picture (masks / video frames excluded)
polyps  = images[images.finding_parent == "polyps"]         # polyps of every dataset (see `procedure`/`region` to separate capsule from colonoscopy)
neg     = images[images.is_negative == True]                # frames without pathology (category normal / background)
boxes   = images[images.has_bbox]                           # bboxes_json holds the per-box labels; inclusive pixel corners
masks   = df[df.source_type == "segmentation_mask"]         # mask_path + parent_image_sample_id -> the image row
train   = images[images.group_quality == "exact"]           # split by group_id (e.g. GroupKFold), NEVER by row
colon   = images[images.region == "lower-gi-tract"]
```
Frames of one clip / sequence / case are near-duplicates: always split by `group_id`; rows with `group_quality` `none` or `proxy` cannot be split leak-safely (see `docs/datasets/`).